# Notebook 04 — Thematic Analysis and Validation



**Duration:** ~35 minutes



In notebook 03 you extracted structured information from legislation — cross-references between Acts. Now you go deeper. Instead of pulling out specific facts, you ask the LLM to identify **themes** — recurring patterns of meaning that run across multiple sections.

You will:

1. Ask the LLM to generate themes from Privacy Act sections using a neutral prompt

2. Repeat the analysis with a **positioned persona** and compare the results

3. Validate both sets of themes against real parliamentary committee domains using **Jaccard similarity**



By the end you will understand how prompt framing shapes thematic output — and why validation is not optional.

## Setup

Run this cell first. It loads your base URL and API key from the `.env` file you set up in notebook 01.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

import os, json, base64, requests, io
from dotenv import load_dotenv
from openai import OpenAI
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load your base URL and API key from the .env file (set up in notebook 01)
load_dotenv(override=True)
if not os.getenv("LLM_BASE_URL") or not os.getenv("LLM_API_KEY"):
    raise RuntimeError("Could not find LLM_BASE_URL or LLM_API_KEY. Check your .env file (see notebook 01).")

client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# Each Dell Pro Max machine serves one model. This is its name.
# It reads both text and images.
MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4"

# Turn off the model's "thinking" step. Answers come back several times faster,
# which matters when many people share one machine.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

print(f"Using model {MODEL}.")
print("Setup complete.")

## Load and prepare the corpus

We use the same dataset as notebook 03 — the **Privacy Act 2020**, downloaded from the NZ legislation website and saved in the workshop's `data` folder. The cell below loads it, parses it into sections, and selects a 20-section sample for analysis.

Why 20 sections? It is enough text for the LLM to find meaningful patterns, but small enough to fit within API limits and run in a few seconds.

In [ ]:
import re

# Load the Privacy Act 2020 XML from the workshop's data folder
# Original source: https://legislation.govt.nz/act/public/2020/31/en/latest.xml
root = etree.parse("../data/privacy-act-2020.xml").getroot()

# Parse each <prov> element into a dictionary
sections = []
for prov in root.findall(".//prov"):
    label_el = prov.find(".//label")
    heading_el = prov.find(".//heading")
    body_el = prov.find(".//prov.body")

    label = label_el.text.strip() if label_el is not None and label_el.text else "?"
    heading = heading_el.text.strip() if heading_el is not None and heading_el.text else ""
    text = "".join(body_el.itertext()).strip() if body_el is not None else ""

    sections.append({"label": label, "heading": heading, "text": text})

print(f"Loaded {len(sections)} sections from the Privacy Act 2020.")

In [ ]:
# Select a 20-section sample and build the corpus text for the LLM
sample_sections = sections[2:22]

# Truncate each section to 400 characters to keep the API call manageable
corpus_lines = []
for s in sample_sections:
    text_preview = s["text"][:400]
    if len(s["text"]) > 400:
        text_preview += "..."
    corpus_lines.append(f"Section {s['label']}: {s['heading']}\n{text_preview}")

corpus_text = "\n\n".join(corpus_lines)

print(f"Corpus: {len(sample_sections)} sections, {len(corpus_text)} characters")
print()
print("Sections included:")
for s in sample_sections:
    print(f"  Section {s['label']}: {s['heading']}")

# Show the first section as an example so you can see what the data looks like
print()
print("Example — first section in the sample:")
print("=" * 60)
ex = sample_sections[0]
print(f"Section {ex['label']}: {ex['heading']}")
print(ex["text"][:500])
if len(ex["text"]) > 500:
    print("...")

## Where this fits in the research workflow

> **Research context:** The published study on NZ legislative networks (Ardekani et al., 2026) used an LLM to generate compact summaries and keywords for each Act — a process they called "semantic enrichment." This replaced noisy raw text with clean thematic descriptions that could be fed into topic modelling. The authors then validated their computationally derived topic clusters by comparing them against real parliamentary committee domains using **Jaccard similarity**. You are about to do the same thing — on a smaller scale, with the same logic.

In this notebook you are doing:
- **Semantic enrichment** (Step 4) — asking the LLM to identify themes and generate keywords from legislative text
- **Analysis** (Step 5) — comparing themes across different prompt framings
- **Interpretation** (Step 6) — validating theme keywords against external reference sets using Jaccard similarity

## Thematic analysis

In this notebook we use theme generation — we give the LLM the text and ask it what themes it finds, without telling it what to look for. This is closer to grounded theory than hypothesis testing.

## Exercise a — Theme generation

We send the 20-section corpus to the LLM with a **neutral prompt** — no persona, no framing, just an instruction to identify themes.

The prompt asks the model to:
- Identify 3 recurring themes
- Focus on substantive themes (legal rights, obligations, powers) rather than procedural boilerplate
- Return each theme with a name, description, and 10 to 15 keywords

Run the cell below. Read the output carefully — you will compare it against a second run in exercise b.

In [ ]:
neutral_response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a qualitative research assistant analysing New Zealand legislation."
        },
        {
            "role": "user",
            "content": f"""Analyse the following sections from New Zealand's Privacy Act 2020.

Identify 5 recurring themes. Focus on substantive legal themes and obligations.
Exclude procedural boilerplate (e.g. "commencement", "interpretation", "repeal").

For each theme, provide:
- A short theme name
- A one-sentence description
- 10 to 15 keywords that characterise the theme

Return your answer as a JSON array:
[
  {{
    "theme": "Theme name",
    "description": "One-sentence description",
    "keywords": ["keyword1", "keyword2", ...]
  }}
]

Return ONLY the JSON array.

Sections:
{corpus_text}"""
        }
    ]
)

neutral_raw = neutral_response.choices[0].message.content.strip()

# Parse the JSON
try:
    neutral_themes = json.loads(neutral_raw)
except json.JSONDecodeError:
    cleaned = neutral_raw.strip("`").strip()
    if cleaned.startswith("json"):
        cleaned = cleaned[4:].strip()
    neutral_themes = json.loads(cleaned)

# Display the themes
print(f"Neutral prompt: {len(neutral_themes)} themes identified")
print("=" * 60)
for i, theme in enumerate(neutral_themes, 1):
    print(f"\n{i}. {theme['theme']}")
    print(f"   {theme['description']}")
    kw_list = ", ".join(theme["keywords"])
    print(f"   Keywords: {kw_list}")

Read through the themes. Some questions to consider:

- Do the themes make sense for a privacy law? Would you have identified similar ones?
- Are the keywords specific enough to be useful, or are they vague?
- Did the model exclude procedural boilerplate as instructed, or did some slip through?
- If you ran this cell again (with `temperature=0`), would you expect the same themes?

Keep these themes in mind. We are about to run the same analysis from a different perspective.

## Exercise b — Persona experiment

In notebook 01 you learned that the **system message** sets the model's role. In exercise a, the role was neutral: "a qualitative research assistant." Now we change it.

A **positioned persona** tells the model to adopt a specific perspective. The exercise is about explictly documenting your positionality. 

Choose one of the personas below (or write your own) and run the cell. The instruction is identical to exercise a.

In [ ]:
# Choose a positioned persona. Uncomment one of the options below,
# or write your own.
#
# Option 1 — Privacy rights advocate:
# my_persona = "You want to point out faults by identifying risks in this legislation."
#
# Option 2 — Indigenous data sovereignty researcher:
# my_persona = "You are a Māori data sovereignty researcher analysing this legislation's implications for indigenous data governance."

# Option 3 - Your own persona (change this):
# my_persona = "You are a grammar checker. Critique the language and structure of this legislation."

positioned_response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": my_persona
        },
        {
            "role": "user",
            "content": f"""Analyse the following sections from New Zealand's Privacy Act 2020.

Identify 5 recurring themes. Focus on substantive legal themes and obligations.
Exclude procedural boilerplate (e.g. "commencement", "interpretation", "repeal").

For each theme, provide:
- A short theme name
- A one-sentence description
- 10 to 15 keywords that characterise the theme

Return your answer as a JSON array:
[
  {{
    "theme": "Theme name",
    "description": "One-sentence description",
    "keywords": ["keyword1", "keyword2", ...]
  }}
]

Return ONLY the JSON array.

Sections:
{corpus_text}"""
        }
    ]
)

positioned_raw = positioned_response.choices[0].message.content.strip()

# Parse the JSON
try:
    positioned_themes = json.loads(positioned_raw)
except json.JSONDecodeError:
    cleaned = positioned_raw.strip("`").strip()
    if cleaned.startswith("json"):
        cleaned = cleaned[4:].strip()
    positioned_themes = json.loads(cleaned)

# Display the themes
print(f"Positioned prompt: {len(positioned_themes)} themes identified")
print(f"Persona: {my_persona}")
print("=" * 60)
for i, theme in enumerate(positioned_themes, 1):
    print(f"\n{i}. {theme['theme']}")
    print(f"   {theme['description']}")
    kw_list = ", ".join(theme["keywords"])
    print(f"   Keywords: {kw_list}")

In [ ]:
# Collect all keywords from each Exercises a and b into a single set
neutral_keywords = set()
for theme in neutral_themes:
    for kw in theme["keywords"]:
        neutral_keywords.add(kw.lower())

positioned_keywords = set()
for theme in positioned_themes:
    for kw in theme["keywords"]:
        positioned_keywords.add(kw.lower())

# Compare the two keyword sets
shared = neutral_keywords & positioned_keywords
neutral_only = neutral_keywords - positioned_keywords
positioned_only = positioned_keywords - neutral_keywords

print(f"Neutral keywords:    {len(neutral_keywords)}")
print(f"Positioned keywords: {len(positioned_keywords)}")
print(f"Shared:              {len(shared)}")
print()

print(f"Keywords in BOTH ({len(shared)}):")
for kw in sorted(shared):
    print(f"  {kw}")

print(f"\nNeutral ONLY ({len(neutral_only)}):")
for kw in sorted(neutral_only):
    print(f"  {kw}")

print(f"\nPositioned ONLY ({len(positioned_only)}):")
for kw in sorted(positioned_only):
    print(f"  {kw}")

In [ ]:
# Optional — save both theme sets as JSON files
with open("themes_neutral.json", "w") as f:
    json.dump(neutral_themes, f, indent=2)

with open("themes_positioned.json", "w") as f:
    json.dump(positioned_themes, f, indent=2)

print("Saved themes_neutral.json and themes_positioned.json")
print("You can find them in the notebooks folder, next to this notebook.")

## Visualising keyword overlap

The text comparison above lists every keyword — useful for detail, but hard to see the big picture. A **Venn diagram** makes the overlap immediately visible:

- **Left circle (blue):** keywords that only appeared in the neutral prompt
- **Right circle (orange):** keywords that only appeared in the positioned prompt
- **Centre overlap:** keywords both prompts agreed on

This gives you an at-a-glance view of how much the persona shifted the LLM's output.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

fig, ax = plt.subplots(figsize=(12, 8))

v = venn2(
    [neutral_keywords, positioned_keywords],
    set_labels=("Neutral prompt", "Positioned prompt"),
    set_colors=("#4A90D9", "#D97B4A"),
    alpha=0.6,
    ax=ax
)

# Replace the default counts with the actual keyword lists
v.get_label_by_id("10").set_text("\n".join(sorted(neutral_only)))
v.get_label_by_id("10").set_fontsize(12)

v.get_label_by_id("11").set_text("\n".join(sorted(shared)))
v.get_label_by_id("11").set_fontsize(12)

v.get_label_by_id("01").set_text("\n".join(sorted(positioned_only)))
v.get_label_by_id("01").set_fontsize(12)

ax.set_title("Keyword overlap: Neutral vs Positioned prompt", fontsize=13, pad=15)
plt.tight_layout()
plt.show()


### What changed?

The instruction was identical. Only the persona changed. Yet the keywords shifted — some appeared, some disappeared.

This is not a flaw. It is the point. A privacy rights advocate and a business compliance officer read the same law differently, just as two human researchers with different backgrounds would.

The methodological questions are:
- **Which prompt is "neutral"?** Is a prompt with no persona truly neutral, or does it carry its own implicit framing?
- **What does your methods section need to say?** If your results depend on the persona, you need to document which one you used and why.

The tautology is deliberate and instructive: a surveillance-focused persona finds surveillance-related keywords. Naming that is the methodological insight, not a flaw to hide.

### Optional: Save your themes

If you want to keep the raw theme data for later — or compare across participants — you can save both sets as JSON files.

Run the cell below. The files appear in the `notebooks` folder, next to this notebook. You can open them from the VS Code Explorer.

## Exercise c — Jaccard validation

You now have two sets of theme keywords — one from a neutral prompt, one from a positioned prompt. But how do you know whether either set is *meaningful*? Do the themes correspond to anything real?

The published study (Ardekani et al., 2026) faced the same question. They validated their computationally derived topic clusters by comparing them against **parliamentary select committee domain keywords** — real institutional categories that the New Zealand Parliament uses to organise its work.

The measure they used is **Jaccard similarity** — a formula that measures the overlap between two sets:

```
Jaccard(A, B) = |A ∩ B| / |A ∪ B|
```

- If the two sets share every element: Jaccard = 1.0
- If they share nothing: Jaccard = 0.0
- A score of 0.1 means 10% overlap

We will compare your theme keywords against keyword sets for six real parliamentary committees. The question: do the LLM's themes align with recognisable policy domains?

In [ ]:
# Jaccard similarity function
def jaccard_similarity(set_a, set_b):
    """Calculate Jaccard similarity between two sets of strings."""
    a = set(w.lower() for w in set_a)
    b = set(w.lower() for w in set_b)
    intersection = a & b
    union = a | b
    if not union:
        return 0.0
    return len(intersection) / len(union)

# Parliamentary select committee keyword sets
# These represent real policy domains used by the NZ Parliament.
COMMITTEE_KEYWORDS = {
    "Justice Committee": [
        "criminal", "justice", "court", "offence", "penalty",
        "prosecution", "enforcement", "police", "imprisonment",
        "tribunal", "legal", "rights", "appeal", "sentence", "conviction"
    ],
    "Finance and Expenditure Committee": [
        "financial", "revenue", "tax", "expenditure", "budget",
        "fiscal", "economic", "payment", "fund", "investment",
        "commercial", "income", "cost", "penalty", "levy"
    ],
    "Governance and Administration Committee": [
        "privacy", "information", "data", "personal", "agency",
        "public", "government", "official", "disclosure", "access",
        "transparency", "accountability", "complaint", "commissioner",
        "record"
    ],
    "Social Services and Community Committee": [
        "social", "community", "welfare", "family", "housing",
        "poverty", "employment", "education", "youth", "elderly",
        "disability", "support", "benefit", "vulnerable", "protection"
    ]
}

print("Jaccard function and committee keywords loaded.")
print(f"{len(COMMITTEE_KEYWORDS)} committees, {sum(len(v) for v in COMMITTEE_KEYWORDS.values())} total keywords.")

In [ ]:
# Compare neutral theme keywords against each committee
print("Neutral themes — Jaccard similarity with each committee:")
print("=" * 60)
neutral_scores = {}
for committee, kws in COMMITTEE_KEYWORDS.items():
    score = jaccard_similarity(neutral_keywords, kws)
    neutral_scores[committee] = score
    bar = "█" * int(score * 40)
    print(f"  {committee:45s} {score:.3f} {bar}")

best_neutral = max(neutral_scores, key=neutral_scores.get)
print(f"\nBest match: {best_neutral} ({neutral_scores[best_neutral]:.3f})")

In [ ]:
# Compare positioned theme keywords against each committee
print("Positioned themes — Jaccard similarity with each committee:")
print("=" * 60)
positioned_scores = {}
for committee, kws in COMMITTEE_KEYWORDS.items():
    score = jaccard_similarity(positioned_keywords, kws)
    positioned_scores[committee] = score
    bar = "█" * int(score * 40)
    print(f"  {committee:45s} {score:.3f} {bar}")

best_positioned = max(positioned_scores, key=positioned_scores.get)
print(f"\nBest match: {best_positioned} ({positioned_scores[best_positioned]:.3f})")

# Side-by-side comparison
print("\n")
print("Side-by-side comparison:")
print("=" * 70)
print(f"  {'Committee':45s} {'Neutral':>8s} {'Persona':>8s} {'Shift':>6s}")
print("-" * 70)
for committee in COMMITTEE_KEYWORDS:
    n = neutral_scores[committee]
    p = positioned_scores[committee]
    diff = p - n
    if diff > 0.005:
        arrow = "↑"
    elif diff < -0.005:
        arrow = "↓"
    else:
        arrow = "–"
    print(f"  {committee:45s} {n:8.3f} {p:8.3f} {arrow:>5s}")

### Does the validation make sense?

The Privacy Act is fundamentally about information privacy — so the **Governance and Administration Committee** should be the strongest match. If it is not, ask yourself why: is it the model's keywords, or the committee's keyword set that is the weak link?

Now look at the shift arrows:
- Did the positioned prompt **increase** alignment with a specific committee?
- If you chose the privacy rights advocate persona, did the Justice Committee score go up (rights, legal)?
- If you chose the business compliance persona, did Finance and Expenditure shift?

> "The paper validated their clusters this way — comparing computationally derived communities against parliamentary committee domains. You just did the same."

The key insight: Jaccard similarity is a simple, transparent, and citable validation method. You do not need machine learning to validate machine learning outputs. You need a good reference set and a clear question.

## Exercise d — Optional: Adversarial prompting

If you have time, try this stretch exercise. Instead of asking the LLM to generate themes, ask it to **challenge** the themes it already produced.

This is **adversarial prompting** — using the model as its own critic. It is a useful validation technique when you do not have an external reference set to compare against.

Run the cell below. It sends your neutral themes back to the model and asks it to find weaknesses in each one.

In [ ]:
# Send the neutral themes back to the model and ask it to critique them
adversarial_response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a critical peer reviewer of qualitative research methods."
        },
        {
            "role": "user",
            "content": f"""A researcher used an LLM to identify the following themes
in sections of New Zealand's Privacy Act 2020:

{json.dumps(neutral_themes, indent=2)}

For each theme, identify one weakness or potential bias in how it was framed.
Be specific — refer to the theme name and keywords.
Keep each critique to 1-2 sentences."""
        }
    ]
)

print("Adversarial critique of neutral themes:")
print("=" * 60)
print(adversarial_response.choices[0].message.content)

## What you accomplished

In this notebook you:

- Generated **inductive themes** from Privacy Act sections using a neutral LLM prompt (**Step 4: Semantic Enrichment**)
- Ran the same analysis with a **positioned persona** and measured how the keyword set shifted (**Step 5: Analysis**)
- Validated both sets of themes against parliamentary committee domains using **Jaccard similarity** (**Step 6: Interpretation**)
- Saw that prompt framing is a **methodological choice** that must be documented, not a setting to ignore

You now have three named validation techniques in your toolkit:

| Technique | What it checks | When you used it |
|-----------|---------------|------------------|
| **Consistency check** | Does the method give the same answer twice? | Notebook 02 (temperature) |
| **Cross-method comparison** | Do two methods agree? | Notebook 02 (Ardekani et al. 2026) |
| **Jaccard validation** | Do the results align with a known reference? | This notebook |

The key insight: the themes an LLM finds depend on how you ask. A neutral prompt and a positioned prompt can read the same text and produce different themes. Neither is wrong — but both need to be validated, and the choice needs to be documented.

**Next up:** Notebook 05 applies the same extraction and validation pipeline to images instead of text. The data type changes. The methodology does not.